# Training a Coordinate Entry Classifier

This notebook fine-tunes a multilingual BERT model (`bert-base-multilingual-cased`) as a binary classifier to detect entries containing geographic coordinates in the *Encyclopédie*.

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from transformers import AutoTokenizer, BertForSequenceClassification, Trainer, TrainingArguments


In [ ]:
# ========================
# Dataset wrapper
# ========================
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        encoding = self.tokenizer(
            text_target=text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# ========================
# Metric function for HuggingFace Trainer
# ========================
def compute_metrics(pred):
    labels = pred.label_ids
    logits = pred.predictions
    preds = logits.argmax(-1)
    probs_pos = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds)
    rec = recall_score(labels, preds)
    f1 = f1_score(labels, preds)
    try:
        roc_auc = roc_auc_score(labels, probs_pos)
        pr_auc = average_precision_score(labels, probs_pos)
    except Exception:
        roc_auc, pr_auc = float('nan'), float('nan')
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "roc_auc": roc_auc, "pr_auc": pr_auc}

# ========================
# Custom Trainer with class weights for imbalance
# ========================
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        if self.class_weights is not None:
            loss_fct = torch.nn.CrossEntropyLoss(
                weight=self.class_weights.to(logits.device)
            )
        else:
            loss_fct = torch.nn.CrossEntropyLoss()
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ========================
# Main Cross-validation loop
# ========================
def cross_validate(df, model_name="camembert-base", n_splits=5, epochs=3, batch_size=8):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    results = []

    # Device selection: Apple Silicon (MPS) or CUDA if available
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

    print(f"Training on device: {device}")

    for fold, (train_idx, val_idx) in enumerate(skf.split(df["text"], df["label"])):
        print(f"===== Fold {fold+1} / {n_splits} =====")

        train_texts, val_texts = df.loc[train_idx, "text"].tolist(), df.loc[val_idx, "text"].tolist()
        train_labels, val_labels = df.loc[train_idx, "label"].tolist(), df.loc[val_idx, "label"].tolist()

        train_dataset = TextDataset(train_texts, train_labels, tokenizer)
        val_dataset = TextDataset(val_texts, val_labels, tokenizer)

        model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)
        model.to(device)

        training_args = TrainingArguments(
            output_dir=f"./results_fold{fold}",
            eval_strategy="epoch",
            save_strategy="epoch",  
            save_total_limit=1,       
            learning_rate=2e-5,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            num_train_epochs=epochs,
            weight_decay=0.01,
            logging_dir=f"./logs_fold{fold}",
            logging_steps=50,
            load_best_model_at_end=True,
            metric_for_best_model="pr_auc", 
            greater_is_better=True,
        )

        # --- class weights (inverse frequency) on the training fold ---
        class_counts = np.bincount(train_labels, minlength=2)
        # Standard weights: N / (K * count_c)
        N = class_counts.sum()
        K = len(class_counts)
        class_weights = torch.tensor([N / (K * c) if c > 0 else 0.0 for c in class_counts], dtype=torch.float)

        trainer = WeightedTrainer(
            class_weights=class_weights,
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=tokenizer,
            compute_metrics=compute_metrics,
        )

        trainer.train()
        metrics = trainer.evaluate()
        print(metrics)
        results.append(metrics)

    # Summary of results
    results_df = pd.DataFrame(results)
    print("===== Average scores =====")
    print(results_df[["eval_accuracy","eval_precision","eval_recall","eval_f1","eval_roc_auc","eval_pr_auc"]].mean())
    return results_df


## Prepare dataset

In [ ]:
df = pd.read_json("../data/diderot_1751_wd.json")
df.head()

,vedette,entreeid,texte,qid,note,coord_extr,qid_region,renvoi,mérid_orig
0,A,v1-9-0,"*​ A, s. petite riviere de France, qui a sa so...",[Q15803785],https://fr.wikisource.org/wiki/Page:Trevoux_-_...,NaN,NaN,NaN,NaN
1,AA,v1-10-0,"*​ AA, s. f. riviere de France, qui prend sa s...",[Q300661],NaN,NaN,NaN,NaN,NaN
2,AACH ou ACH,v1-12-0,"*​ AACH ou ACH, s. f. petite ville d’Allemagne...",[Q62158],,[[47 55' N 26 57' E]],NaN,NaN,NaN
3,AAHUS,v1-13-0,"*​ AAHUS, s. petite ville d’Allemagne dans le ...",[Q14888],,[[52 10' N 24 36' E]],NaN,NaN,NaN
4,AAR,v1-15-0,"*​ AAR, s. grande riviere qui a sa source proc...",[Q153394],,NaN,NaN,NaN,NaN


In [4]:
df_true = pd.read_json("data/edda_coordinata.json")
df_true['label'] = 1
df_true.head()

,id-enccre,headword,text,coordinates,meridian,label
0,v1-1013-0,AIX,"*​ AIX, (Géog.)​ ville de France en Provence, ...","[['43 31\' 35"" N 23 6\' 34"" E']]",NaN,1
1,v1-1013-1,Aix,"*​ Aix, (Géog.)​ ville de Savoye sur le lac de...","[[""45 40' N 23 34' E""]]",NaN,1
2,v1-1013-3,Aix-la-Chapelle,"*​ Aix-la-Chapelle, (Géog.)​ ville d’Allemagne...","[[""51 55' N 23 55' E""]]",NaN,1
3,v1-1015-0,AKISSAR ou AK-HISSAR,"*​ AKISSAR ou AK-HISSAR, (Géog.)​ ville d’Asie...","[[""38 50' N 46 E""]]",NaN,1
4,v1-1021-1,Alais,"*​ Alais, (Géog.)​ ville de France dans le bas...","[[""44 8' N 21 32' E""]]",NaN,1


In [5]:
# get rows from df not in df_true
df_false = df[~df["entreeid"].isin(df_true["id-enccre"])]
df_false['label'] = 0
df_false.rename(columns={"texte": "text"}, inplace=True)
df_false.head()

/tmp/ipykernel_283/1766922362.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_false['label'] = 0
/tmp/ipykernel_283/1766922362.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_false.rename(columns={"texte": "text"}, inplace=True)


,vedette,entreeid,text,qid,note,coord_extr,qid_region,renvoi,mérid_orig,label
0,A,v1-9-0,"*​ A, s. petite riviere de France, qui a sa so...",[Q15803785],https://fr.wikisource.org/wiki/Page:Trevoux_-_...,NaN,NaN,NaN,NaN,0
1,AA,v1-10-0,"*​ AA, s. f. riviere de France, qui prend sa s...",[Q300661],NaN,NaN,NaN,NaN,NaN,0
4,AAR,v1-15-0,"*​ AAR, s. grande riviere qui a sa source proc...",[Q153394],,NaN,NaN,NaN,NaN,0
5,Aar,v1-15-1,"*​ Aar, s. riviere d’Allemagne qui a sa source...",[Q153394],https://quod.lib.umich.edu/cgi/t/text/text-idx...,NaN,NaN,NaN,NaN,0
6,AA ou AAS,v1-16-0,"*​ AA ou AAS, s. ou Fontaine des Arquebusades....",[Q89345055],Eaux-Bonnes et http://infoterre.brgm.fr/rappor...,NaN,NaN,NaN,NaN,0


In [6]:
df_true = df_true[["text", "label"]]
df_false = df_false[["text", "label"]]

In [7]:
# concatenate true and false dataframes and shuffle
df_final = pd.concat([df_true, df_false], ignore_index=True)
df_final = df_final.sample(frac=1).reset_index(drop=True)
df_final

,text,label
0,"LOCARNO, (Géog.)​ en latin moderne Locarnum, l...",1
1,"Tyndarium promontorium, (Géog. anc.)​ promonto...",0
2,"ZAMBALES, (Géog. mod.)​ peuples des Philippine...",0
3,"MATARO, (Géog.)​ petite ville d’Espagne, dans ...",1
4,"VALROMEY, (Géog. mod.)​ petit pays de France, ...",0
...,...,...
15273,"MUYDEN, (Géog.)​ petite ville des Provinces-Un...",1
15274,"Coule, (Géog. mod.)​ petite ville de Hongrie, ...",0
15275,"CANCHES, (Géog.)​ Sauvages de l’Amérique mérid...",0
15276,"JANJA, (Géog.)​ fleuve de la Sibérie septentri...",0


In [ ]:
id2label = {0: "Negative", 1: "Positive"}
label2id = {"Negative": 0, "Positive": 1}

In [8]:
cross_validate(df_final, model_name="google-bert/bert-base-multilingual-cased", n_splits=5, epochs=4, batch_size=8)


Training on device: cuda
===== Fold 1 / 5 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_283/483346056.py:55: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc,Pr Auc
1,0.123900,0.051802,0.990838,0.994687,0.976017,0.985263,0.999398,0.998871
2,0.000600,0.047094,0.990510,0.988445,0.981230,0.984825,0.999651,0.999280
3,0.013400,0.046214,0.992474,0.989540,0.986444,0.987990,0.999738,0.999462
4,0.000400,0.040816,0.992801,0.988530,0.988530,0.988530,0.999714,0.999408


{'eval_loss': 0.04621385037899017, 'eval_accuracy': 0.9924738219895288, 'eval_precision': 0.9895397489539749, 'eval_recall': 0.986444212721585, 'eval_f1': 0.9879895561357702, 'eval_roc_auc': 0.9997384415792361, 'eval_pr_auc': 0.9994617121527165, 'eval_runtime': 26.6927, 'eval_samples_per_second': 114.488, 'eval_steps_per_second': 14.311, 'epoch': 4.0}
===== Fold 2 / 5 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_283/483346056.py:55: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(**kwargs)


...

{'eval_loss': 0.03727385401725769, 'eval_accuracy': 0.9918166939443536, 'eval_precision': 0.9854469854469855, 'eval_recall': 0.9885297184567258, 'eval_f1': 0.986985944820406, 'eval_roc_auc': 0.9996572248445821, 'eval_pr_auc': 0.9993091036364589, 'eval_runtime': 26.3874, 'eval_samples_per_second': 115.775, 'eval_steps_per_second': 14.477, 'epoch': 4.0}
===== Fold 5 / 5 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_283/483346056.py:55: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc,Pr Auc
1,0.029500,0.061263,0.989525,0.990466,0.975992,0.983176,0.999299,0.998610
2,0.030200,0.054051,0.990507,0.986387,0.983299,0.984841,0.998901,0.995888
3,0.024900,0.043356,0.991489,0.986430,0.986430,0.986430,0.998917,0.995649
4,0.007500,0.046971,0.992799,0.987500,0.989562,0.988530,0.998901,0.995220


{'eval_loss': 0.061262890696525574, 'eval_accuracy': 0.9895253682487725, 'eval_precision': 0.9904661016949152, 'eval_recall': 0.975991649269311, 'eval_f1': 0.9831756046267087, 'eval_roc_auc': 0.9992991279917727, 'eval_pr_auc': 0.9986103455791102, 'eval_runtime': 26.683, 'eval_samples_per_second': 114.493, 'eval_steps_per_second': 14.316, 'epoch': 4.0}
===== Moyenne des scores =====
eval_accuracy     0.992014
eval_precision    0.988304
eval_recall       0.986231
eval_f1           0.987255
eval_roc_auc      0.999644
eval_pr_auc       0.999273
dtype: float64


,eval_loss,eval_accuracy,eval_precision,eval_recall,eval_f1,eval_roc_auc,eval_pr_auc,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch
0,0.046214,0.992474,0.989540,0.986444,0.987990,0.999738,0.999462,26.6927,114.488,14.311,4.0
1,0.048650,0.992147,0.986472,0.988530,0.987500,0.999734,0.999443,27.0895,112.811,14.101,4.0
2,0.025460,0.994110,0.989594,0.991658,0.990625,0.999789,0.999539,26.2149,116.575,14.572,4.0
3,0.037274,0.991817,0.985447,0.988530,0.986986,0.999657,0.999309,26.3874,115.775,14.477,4.0
4,0.061263,0.989525,0.990466,0.975992,0.983176,0.999299,0.998610,26.6830,114.493,14.316,4.0


In [ ]:
# Train the final model on the entire dataset
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-multilingual-cased")
model = BertForSequenceClassification.from_pretrained("google-bert/bert-base-multilingual-cased", num_labels=2)

# Device selection: Apple Silicon (MPS) or CUDA if available
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

print(f"Training final model on device: {device}")

full_dataset = TextDataset(df_final["text"].tolist(), df_final["label"].tolist(), tokenizer)

training_args = TrainingArguments(
    output_dir="./final_model_results",
    eval_strategy="no", # No evaluation during training of the final model
    save_strategy="epoch",
    save_total_limit=1,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./final_model_logs",
    logging_steps=50,
)

# --- class weights (inverse frequency) on the full dataset ---
class_counts = np.bincount(df_final["label"].tolist(), minlength=2)
N = class_counts.sum()
K = len(class_counts)
class_weights = torch.tensor([N / (K * c) if c > 0 else 0.0 for c in class_counts], dtype=torch.float)

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=full_dataset,
    tokenizer=tokenizer,
)

trainer.train()

trainer.config.id2label = id2label
trainer.config.label2id = label2id

# Save the final model
model_save_path = "final_bert_model"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"Final model saved to {model_save_path}")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training final model on device: cuda


/tmp/ipykernel_283/483346056.py:55: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(**kwargs)


Step,Training Loss
50,0.319600
100,0.162800
150,0.078400
200,0.061400
250,0.116300
300,0.099700
350,0.091500
400,0.077900
450,0.090100
500,0.077000


Final model saved to final_bert_model


In [6]:
from transformers import pipeline
binary_classifier = pipeline("text-classification", model="GEODE/bert-base-multilingual-cased-binary-classifier-edda-coords")

text = "* AACH ou ACH, s. f. petite ville d'Allemagne dans le cercle de Souabe, près de la source de l'Aach. Long. 26. 57. lat. 47. 55."
binary_classifier(text)

Device set to use mps:0


[{'label': 'Positive', 'score': 0.9999674558639526}]

In [7]:
text = "* AACH ou ACH, s. f. petite ville d'Allemagne dans le cercle de Souabe, près de la source de l'Aach."
binary_classifier(text)

[{'label': 'Negative', 'score': 0.9999685287475586}]

In [8]:
res = binary_classifier(text)
res[0]['label']

'Negative'